# Self-Supervised Pretraining using Vision Transformer

---

## Notebook 2: Contrastive Representation Learning

This notebook implements:

- Dual Augmentation Strategy (SimCLR-style)
- Vision Transformer Encoder
- Projection Head
- NT-Xent Contrastive Loss
- SSL Training Loop
- Saving Pretrained Encoder

Output:
models/ssl_encoder.pth

## 1. Objective

Supervised models often overfit to labeled datasets.

To improve generalization, we first train the encoder using:

Self-Supervised Contrastive Learning

The goal is to:

- Learn invariant feature embeddings
- Improve robustness to domain shift
- Reduce label dependency

## Import Libraries

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
import cv2
import glob
import numpy as np
from tqdm import tqdm

import albumentations as A
from albumentations.pytorch import ToTensorV2

E:\conda_envs\ml_forever\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Device Configuration

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


## Load Training Image Paths

In [3]:
import os
import glob

DATA_DIR = r"D:\Downloads Alt\archive\real_vs_fake\real-vs-fake"

train_real = glob.glob(os.path.join(DATA_DIR, "train/real/*.jpg"))
train_fake = glob.glob(os.path.join(DATA_DIR, "train/fake/*.jpg"))

ssl_image_paths = (train_real + train_fake)[:1000]
print("Using subset size:", len(ssl_image_paths))

print("Train Real:", len(train_real))
print("Train Fake:", len(train_fake))
print("Total SSL Images:", len(ssl_image_paths))

Using subset size: 1000
Train Real: 50000
Train Fake: 50000
Total SSL Images: 1000


## 2. Dual Augmentation Strategy

For each image:
- Generate two random augmented views
- Pass both through encoder
- Pull positive pairs closer
- Push negatives apart

In [4]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

ssl_transform = A.Compose([
    A.RandomResizedCrop(size=(224, 224)),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(p=0.8),
    A.GaussianBlur(p=0.5),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2()
])

### SSL Dataset Class

In [5]:
class SSLDataset(Dataset):
    def __init__(self, image_paths, transform):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        view1 = self.transform(image=img)["image"]
        view2 = self.transform(image=img)["image"]

        return view1, view2

### SSL DataLoader

In [6]:
ssl_dataset = SSLDataset(ssl_image_paths, ssl_transform)
ssl_loader = DataLoader(ssl_dataset, batch_size=4, shuffle=True)

## 3. SSL Model Architecture

Backbone:
Vision Transformer (ViT-Base Patch16-224)

Projection Head:
MLP: 768 → 512 → 128

The projection head is used only during SSL training.

In [7]:
class SSLViT(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = timm.create_model(
            "vit_tiny_patch16_224", 
            pretrained=True,
            num_classes=0
        )

        feature_dim = self.encoder.num_features  
        self.projector = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 128)
        )

    def forward(self, x):
        features = self.encoder(x)
        projection = self.projector(features)
        return projection

## 4. NT-Xent Contrastive Loss

Encourages:
- Similar embeddings for augmented pairs
- Dissimilar embeddings for different images

In [8]:
def nt_xent_loss(z1, z2, temperature=0.5):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    representations = torch.cat([z1, z2], dim=0)
    similarity_matrix = torch.matmul(representations, representations.T)

    batch_size = z1.size(0)

    mask = torch.eye(2*batch_size, dtype=torch.bool).to(device)
    similarity_matrix = similarity_matrix[~mask].view(2*batch_size, -1)

    positives = torch.cat([
        torch.sum(z1*z2, dim=-1),
        torch.sum(z2*z1, dim=-1)
    ], dim=0)

    positives = positives.unsqueeze(1)

    logits = torch.cat([positives, similarity_matrix], dim=1)
    logits /= temperature

    labels = torch.zeros(2*batch_size).long().to(device)

    loss = F.cross_entropy(logits, labels)
    return loss

## Model & Optimizer

In [9]:
ssl_model = SSLViT().to(device)
optimizer = torch.optim.AdamW(ssl_model.parameters(), lr=1e-4)

In [17]:
#pip install hf_xet

## SSL Training Loop

In [10]:
epochs = 1

for epoch in range(epochs):
    ssl_model.train()
    total_loss = 0

    for x1, x2 in tqdm(ssl_loader):
        x1 = x1.to(device)
        x2 = x2.to(device)

        z1 = ssl_model(x1)
        z2 = ssl_model(x2)

        loss = nt_xent_loss(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}] SSL Loss: {total_loss/len(ssl_loader):.4f}")

100%|████████████████████████████████████████████████████████████████████████████████| 250/250 [06:14<00:00,  1.50s/it]

Epoch [1/1] SSL Loss: 1.5667


### Save Pretrained Encoder

In [11]:
import os

os.makedirs("models", exist_ok=True)

torch.save(
    ssl_model.encoder.state_dict(),
    "models/ssl_encoder.pth"
)

print("SSL Encoder Saved Successfully.")

SSL Encoder Saved Successfully.


## SSL Pretraining Completed

We have successfully:

- Learned invariant feature representations
- Trained a Vision Transformer encoder
- Saved pretrained encoder weights

Next Step:
Supervised Fine-Tuning using labeled data.